In [1]:
import numpy as np, pandas as pd
from scipy import stats
seeds = [110, 111, 112, 113, 114]
rows = []
for s in seeds:
    r = pd.read_csv(f"../outputs/tables/extra_metrics_seed{s}.csv")
    rows.append(dict(seed=s, hospitals=len(r), nonlocal_per_hospital_alpha=int((r.sel_alpha005 > 0).sum()),
                     nonlocal_simultaneous=int((r.sel_bonferroni > 0).sum()),
                     eo_gap_local=r.eo_local.mean(), eo_gap_pooled=r.eo_pooled.mean(), eo_gap_certified=r.eo_certified.mean(),
                     median_loss_diff_pooled=r.diff_pooled.median(), wilcoxon_p_pooled=stats.wilcoxon(r.diff_pooled).pvalue,
                     median_loss_diff_certified=r.diff_certified.median(),
                     wilcoxon_p_certified=stats.wilcoxon(r.diff_certified).pvalue if (r.diff_certified != 0).any() else 1.0))
T = pd.DataFrame(rows); T.loc["mean"] = T.mean(numeric_only=True)
T.round(5).to_csv("../outputs/tables/T_extra_metrics.csv"); print(T.round(5).to_string())

comm = pd.DataFrame([
    ("FedAvg / FedProx, 30 rounds (logistic regression, 53 coefficients, 8-byte floats, up and down)", 25440, 848),
    ("Fine-tuning (one download of the pooled model)", 424, 424),
    ("Certificate, run at the hospital on its own certification split (no extra communication beyond the model download)", 424, 424),
    ("Certificate, central version (2 scalars per candidate and group; 4 candidates, 8 groups)", 512, 512)],
    columns=["method", "bytes_per_hospital_total", "bytes_per_round_or_message"])
comm.to_csv("../outputs/tables/T_communication.csv", index=False); print(); print(comm.to_string(index=False))

ledger = pd.DataFrame([
 ("0", "Pilot: tolerance grid, clipping, calibration power checks", "Exploratory; not reported as results"),
 ("100", "First loss-only run of the final code", "Exploratory (amendment 2)"),
 ("101-104", "Loss-only harm study and baselines (pooled, blends, certified, FedAvg, FedProx, fine-tuning, recalibration)", "Original confirmatory (amendments 2, 3)"),
 ("105-109", "Certified fine-tuned variant and baseline table (loss only)", "Confirmatory for amendment 4"),
 ("110-114", "Full certificate (loss and calibration) and all later analyses on the same splits: subgroup, intersectional, APR, temporal, size cutoffs, bound choice, extra metrics, participation, Ditto and q-FFL, model class (MLP 111-114), leave-one-hospital-out", "Headline confirmatory (amendments 5-8, 10, 11, 13-18); same splits across analyses, so not independent"),
 ("200-204", "ICU 2012 stress test and ICU diagnostic", "Amendments 12 and 19; resamples splits of the same 12,000 stays"),
 ("simulation 0-199", "Simulation with the actual certificate", "Amendment 9")],
 columns=["seeds", "used for", "status"])
ledger.to_csv("../outputs/tables/T4c_seed_ledger.csv", index=False)
print(); print(ledger.to_string(index=False))

       seed  hospitals  nonlocal_per_hospital_alpha  nonlocal_simultaneous  eo_gap_local  eo_gap_pooled  eo_gap_certified  median_loss_diff_pooled  wilcoxon_p_pooled  median_loss_diff_certified  wilcoxon_p_certified
0     110.0      126.0                         57.0                    6.0       0.11192        0.10105           0.10668                  0.00209                0.0                         0.0               0.06024
1     111.0      126.0                         56.0                    6.0       0.11201        0.10551           0.10897                  0.00171                0.0                         0.0               0.32765
2     112.0      126.0                         55.0                    5.0       0.11229        0.10910           0.11345                  0.00184                0.0                         0.0               0.09056
3     113.0      126.0                         56.0                    6.0       0.11575        0.10017           0.11063               